# C2 · Encoding a nominal with integers: the invented order

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Show that a linear model with a nominal variable encoded as integers depends on the **arbitrary order** of the
categories, while the one-hot does not, and that a tree copes because it isolates the categories with its splits.

## Dataset

Synthetic: 1200 rows with a nominal variable `zone` (8 levels, A-H) that shifts the target by `+6, −6, +5, −5, +7, −7, +4, −4`
(a **non-monotonic** effect in alphabetical order: high and low zones alternate), a numeric feature `x` (`+2·x`) and noise. The function `evaluate(encoder,
model)` builds the pipeline and returns the mean cross-validated R².

## Educational purpose

Get a hands-on feel for why "encode with 0, 1, 2, ..." is not neutral: it hands a linear model a monotonic relationship
between code and target that does not exist. You will see that the R² changes a lot **just by changing the order** of the categories.

## Task

1. Compute `r2_ord_lin` (`OrdinalEncoder()` with alphabetical order + `LinearRegression`), `r2_ohe_lin` (`OneHotEncoder(sparse=False)` + `LinearRegression`), `r2_ord_tree` and `r2_ohe_tree` (`DecisionTreeRegressor(max_depth=8, random_state=0)` in place of the regression).
2. For 200 random permutations of the order of the categories (`np.random.default_rng(s).permutation(categories)` for `s = 0..199`) compute the R² of the linear regression with `OrdinalEncoder(categories=[order])` and save it in `r2_perm`.
3. Compute `spread = r2_perm.max() − r2_perm.min()` and `r2_perm_max`.
4. Choose in `answer` ("a", "b" or "c") the correct explanation:
   - **a**: the OrdinalEncoder assigns codes at random and the result changes at every run;
   - **b**: a linear model sees the code as a quantity: it imposes that the effect grows (or falls) linearly along the chosen order, and with a non-monotonic effect no order works well;
   - **c**: linear regression does not accept categorical variables.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
n = 1200
effect = {"A": 6, "B": -6, "C": 5, "D": -5, "E": 7, "F": -7, "G": 4, "H": -4}      # high and low alternate: non-monotonic
df = pd.DataFrame({"zone": rng.choice(list(effect), n), "x": rng.normal(size=n)})
y = df.zone.map(effect).to_numpy() + 2 * df.x.to_numpy() + rng.normal(size=n)
categories = np.array(sorted(effect))


def evaluate(encoder, model):
    pipe = make_pipeline(ColumnTransformer([("zone", encoder, ["zone"])], remainder="passthrough"), model)
    return cross_val_score(pipe, df[["zone", "x"]], y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()

## Your work

In [ ]:
r2_ord_lin = ...
r2_ohe_lin = ...
r2_ord_tree = ...
r2_ohe_tree = ...
print(f"linear:  ordinal {r2_ord_lin:.3f} | one-hot {r2_ohe_lin:.3f}")
print(f"tree  :  ordinal {r2_ord_tree:.3f} | one-hot {r2_ohe_tree:.3f}")

In [ ]:
r2_perm = []
for s in range(200):
    ...
r2_perm = np.array(r2_perm)
spread = ...
r2_perm_max = ...

In [ ]:
answer = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`evaluate(OrdinalEncoder(), LinearRegression())`. For a permutation: `order = np.random.default_rng(s).permutation(categories)` and `evaluate(OrdinalEncoder(categories=[order]), LinearRegression())`.'):
    check(r2_ohe_lin > 0.85, "the one-hot with the linear model recovers the effect of the zones (R2 > 0.85)", "the linear one-hot should have a high R2")
    check(r2_ord_lin < 0.3, "with integer codes the linear model fails (R2 < 0.3)", "with alphabetical order the effect is not monotonic")
    check(r2_ohe_lin > r2_ord_lin + 0.5, "huge difference between the two encodings, on the same data", "r2_ohe_lin must exceed r2_ord_lin by more than 0.5")
    check(r2_ord_tree > 0.8, "the tree copes even with integer codes (it isolates the categories with its splits)", "a depth 8 tree should have R2 > 0.8")
    check(len(r2_perm) == 200 and spread > 0.15, "changing ONLY the order of the categories the R2 varies a lot", "the spread between the permutations should exceed 0.15")
    check(r2_ohe_lin > r2_perm_max + 0.05 and np.median(r2_perm) < 0.3,
          "the one-hot beats even the best random order, and a typical order is terrible (median below 0.3)",
          "the one-hot must exceed the maximum of the permutations, whose median is low")
    check(answer == "b", "correct explanation", "reread: what does a linear model assume about the numeric code?")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_ordinal_on_nominal_variable.ipynb`. Open it after you pass the check, or after you have really tried.